In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal
import certifi
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from functools import cache
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.rate_limiters import InMemoryRateLimiter

In [ ]:
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
GENAI_API_KEY = os.getenv("GOOGLE_API_KEY")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

if NVIDIA_API_KEY:
    print("NVIDIA API key found")
else:
    print("No NVIDIA API key found. Add NVIDIA_API_KEY to the .env file")
openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = openrouter.chat.completions.create(model="openrouter/free", messages=messages)
print(f"Answered by {response.model}")
display(Markdown(f"**Response:** {response.choices[0].message.content}"))

In [ ]:
DATA = Path("./data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

In [ ]:
claims[0]

In [ ]:
labels[claims[0]["claim_id"]]

In [ ]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "claim_type": claim["claim_type"],
        "amount_claimed": claim["amount_claimed"],
        "difficulty": label["difficulty"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

In [ ]:
home_alone = [claim for claim in claims if claim["claim_id"] == "CLM-1015"][0]
decision = labels["CLM-1015"]

display(Markdown(f"""**{home_alone['summary']}**

{home_alone['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

In [ ]:
# Load the whole queue, in case TINY is True
queue, queue_labels = get_sample_data()
hangover = [claim for claim in queue if claim["claim_id"] == "CLM-1010"][0]
decision = queue_labels["CLM-1010"]

display(Markdown(f"""**{hangover['summary']}**

{hangover['story']}

**Handler:** {decision['decision']} ${decision['payout']:,.2f}. {decision['reason']}
"""))

In [ ]:
fig = px.histogram(data, x="claim_type", color="decision", title="Claims by type")
fig.show()

In [ ]:
fig = px.histogram(
    data,
    x="difficulty",
    color="decision",
    category_orders={"difficulty": ["routine", "hard", "manual"]},
    title="Claims by difficulty",
)
fig.show()

In [ ]:
class ClaimDecision(BaseModel):
    rationale: str = Field(description="2-3 sentences to explain the decision along with the relevant section of the policy")
    decision: Literal["approve", "decline", "refer"]
    payout: float = Field(description="The amount to pay in US dollars, or 0 for a decline or a referral")

In [ ]:

INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

SYSTEM_PROMPT = f"""{INSTRUCTIONS}
{policy_wording}
"""

def make_user_prompt(claim):
    policy = policies[claim["policy_id"]]
    return (
        "# Policy record\n"
        f"{json.dumps(policy, indent=2)}\n\n"
        "# Claim\n"
        f"{json.dumps(claim, indent=2)}\n"
    )

def make_messages(claim):
    """Build the messages for one claim: the task and the wording, then the policy record and the claim."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": make_user_prompt(claim)},
    ]

In [ ]:
@cache
def gemini(model):
    """One structured-output client per model, built once."""
    llm = ChatGoogleGenerativeAI(
        model=model,
        google_api_key=GENAI_API_KEY,
        temperature=0.3,
        timeout=60,
        max_retries=2,
    )
    return llm.with_structured_output(ClaimDecision)

@cache
def nvidia(model):
    client = ChatNVIDIA(
        model=model,
        api_key=NVIDIA_API_KEY,
        temperature=0.5,
        top_p=1,
        timeout=60,
        max_retries=1
    )
    return client.with_structured_output(ClaimDecision)


def failed(claim, model, start, why):
    return {"model": model, "technique": "LLM", "claim_id": claim["claim_id"],
            "decision": None, "payout": 0, "rationale": f"FAILED: {why}"[:300],
            "seconds": round(time.time() - start, 1)}

PROVIDERS = {"openrouter", "google", "nvidia"}

def decide(claim, model, provider):
    if provider not in PROVIDERS:
        raise ValueError(f"unknown provider: {provider}")
    start = time.time()
    messages = make_messages(claim)
    try:
        if provider == "openrouter":
            response = openrouter.chat.completions.parse(
                model=model, messages=messages, response_format=ClaimDecision,
                extra_body={"provider": {"require_parameters": True}},
            )
            answer = response.choices[0].message.parsed
        elif provider == "google":
            answer = gemini(model).invoke(messages)
        else:
            answer = decide_nvidia(model, messages)
    except Exception as e:
        return failed(claim, model, start, repr(e))

    if answer is None:
        return failed(claim, model, start, "empty structured output")

    return {
        "model": model, "technique": "LLM", "claim_id": claim["claim_id"],
        "decision": answer.decision, "payout": answer.payout,
        "rationale": answer.rationale, "seconds": round(time.time() - start, 1),
    }

In [ ]:
nvidia_client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
    timeout=60,
    max_retries=1,
)

def decide_nvidia(model, messages):
    schema = json.dumps(ClaimDecision.model_json_schema())
    messages = messages + [{
        "role": "user",
        "content": f"Reply with only a JSON object matching this schema, no other text:\n{schema}",
    }]
    response = nvidia_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.3,
        max_tokens=16384,
        timeout=180,
    )
    choice = response.choices[0]
    if choice.finish_reason == "length":
        raise RuntimeError("output truncated: raise max_tokens or use a non-reasoning model")
    text = (choice.message.content or "").strip()
    start, end = text.find("{"), text.rfind("}")   # drop code fences or stray text
    return ClaimDecision.model_validate_json(text[start:end + 1])

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def run(model, claims, labels, provider):
    """Run the model on the claims and compare against the labels."""
    results = []
    for claim in claims:
        answer = decide(claim, model, provider)
        label = labels[claim["claim_id"]]
        results.append({**answer, "difficulty": label["difficulty"], "right": is_right(answer, label)})
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    print(f"{model}: {right} of {len(results)} right in {seconds:.0f} seconds")
    return results

In [ ]:
free_results = run("minimaxai/minimax-m3.1-flash", claims, labels, "nvidia")

In [ ]:
scored = pd.DataFrame(free_results)
scored.pivot_table(index="model", columns="difficulty", values="right", aggfunc="sum")

In [ ]:
def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


show_misses(free_results, labels)

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

messages = [
    {"role": "system", "content": "You are a helpful assistant who answers in the style of a pirate."},
    {"role": "user", "content": "What is the meaning of the universe?"},
]

response = llm.invoke(messages)
display(Markdown(response.content))

In [53]:
recorded = {}

@tool
def get_policy(policy_id: str) -> str:
    """Return the policy record for a given policy ID."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"Policy {policy_id} not found"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision for a given claim ID."""
    recorded[claim_id] = {
        "decision": decision,
        "payout": payout,
        "rationale": rationale,
    }
    return f"Recorded decision for {claim_id}: {decision} ${payout:,.2f}. {rationale}"

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

In [64]:
rate_limiter = InMemoryRateLimiter(requests_per_second=0.2, check_every_n_seconds=0.1, max_bucket_size=1)
llm_nvidia = ChatNVIDIA(
    # model="minimaxai/minimax-m3.1-flash",
    # model="moonshotai/kimi-k3.1-pro",
    model="stepfun-ai/step-5-preview",
    api_key=NVIDIA_API_KEY,
    temperature=0.3,
    max_completion_tokens=16384,
    rate_limiter=rate_limiter,
    timeout=180,
)

/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found stepfun-ai/step-5-preview in available_models, but type is unknown and inference may fail.
  warnings.warn(


In [65]:

AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

{policy_wording}
"""

def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

            {json.dumps(claim, indent=2)}
            """

agent = create_agent(
    model=llm_nvidia,
    tools=[get_policy, record_decision],
    system_prompt=AGENT_SYSTEM_PROMPT
)

In [66]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[1])}]})

for message in result["messages"][1:]:
    message.pretty_print()

/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(


================================== Ai Message ==================================
Tool Calls:
  get_policy (call-cf1fe899-7983-4c9f-88cb-add5db7b27a7)
 Call ID: call-cf1fe899-7983-4c9f-88cb-add5db7b27a7
  Args:
    policy_id: POL-2015
================================= Tool Message =================================
Name: get_policy

{
  "policy_id": "POL-2015",
  "customer_id": "CUS-3015",
  "plan": "first",
  "add_ons": [],
  "purchased_on": "2026-04-02",
  "sold_through": "Meridian Air checkout",
  "trip": {
    "destination": "Paris, France",
    "departs_on": "2026-06-20",
    "returns_on": "2026-06-30",
    "trip_cost": 9600.0
  },
  "travelers": [
    "Kate Callister",
    "Peter Callister",
    "Megan Callister",
    "Jeff Callister",
    "Linnie Callister",
    "Kevin Callister"
  ],
  "status": "active"
}
================================== Ai Message ==================================
Tool Calls:
  record_decision (call-9c140601-886f-4fe5-bc62-872360e19527)
 Call ID: call-9c1406

In [67]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke(
        {"messages": [{"role": "user", "content": make_claim_prompt(claim)}]},
        config={"recursion_limit": 12},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": "openrouter/free",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "seconds": round(time.time() - start, 1),   
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))
    print(f"Processed claim {claim['claim_id']}, recorded {len(recorded)} of {len(claims)}")
    time.sleep(2)

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(


Processed claim CLM-1007, recorded 1 of 5


/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(


Processed claim CLM-1015, recorded 2 of 5


/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(


Processed claim CLM-1017, recorded 3 of 5


/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(


Processed claim CLM-1020, recorded 4 of 5


/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(
/home/ayush_thakur/agents/.venv/lib/python3.13/site-packages/langchain_nvidia_ai_endpoints/chat_models.py:1158: UserWarning: Model 'stepfun-ai/step-5-preview' is not known to support tools. Your tool binding may fail at inference time.
  warnings.warn(


Processed claim CLM-1021, recorded 5 of 5
5 of 5 claims have a recorded decision


In [68]:
import requests, os

r = requests.post(
    "https://integrate.api.nvidia.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {os.getenv('NVIDIA_API_KEY')}"},
    json={"model": "minimaxai/minimax-m3.1-flash",
          "messages": [{"role": "user", "content": "Say hi"}],
          "max_tokens": 20},
    timeout=60,
)
print(r.status_code)
print(dict(r.headers))
print(r.text[:500])

200
{'Date': 'Tue, 24 Nov 2026 08:39:48 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'Access-Control-Expose-Headers': 'nvcf-reqid', 'Nvcf-Reqid': '783be010-b1df-4570-a0a2-680742477f24', 'Nvcf-Status': 'fulfilled', 'Server': 'Python/3.12 aiohttp/3.14.5', 'Vary': 'Origin', 'X-Request-Id': '3dc4d10c-6834-4847-ac37-2824be4a19c6'}
{"id":"chatcmpl-c8dc0d7e-b608-46cb-b1bb-e10e95053187","choices":[{"index":0,"message":{"content":null,"refusal":null,"role":"assistant","reasoning_content":"The user is simply asking me to say hi. I should respond in a friendly manner. This"},"finish_reason":"length","logprobs":null}],"created":1795509587,"model":"minimaxai/minimax-m3.1-flash","object":"chat.completion","usage":{"prompt_tokens":180,"completion_tokens":20,"total_tokens":200,"prompt_tokens_details":{"cached_tokens":128},"completio


In [69]:
pd.DataFrame(free_results + agent_results).groupby(["model", "technique"], sort=False)[["right", "seconds"]].sum()

,,right,seconds
model,technique,,
minimaxai/minimax-m3.1-flash,LLM,3,67.8
openrouter/free,Agent,3,430.2


In [70]:
show_misses(agent_results, labels)

**CLM-1020, manual**: refer $0.00, where the handler said approve $240.00

- Model: Claim is for prescription sunglasses ($240) taken by a macaque at Uluwatu Temple on 2026-08-19, within the covered trip (2026-08-16 to 2026-08-28) on the First plan. Two issues need a colleague's review: (1) Section 5 requires theft to be reported to local police within 24 hours and a copy of the police report to be attached; the customer attempted to report but police declined to file a report about the monkeys, so the required police report is missing and only a tour guide's letter was provided as a substitute. (2) The wording does not settle whether a macaque taking the sunglasses constitutes covered "theft" (triggering the police report requirement) or a covered "loss" of personal belongings. If it is treated as a covered loss, the $240 replacement cost is within the First plan's $500 per-item limit with a $0 deductible, but that determination should be made by a colleague.
- Handler: The manual treats a loss to a wild animal as accidental loss. It is not theft, so no police report is needed, and a wild animal is not vermin or a pet. The guide's letter is acceptable evidence. The sunglasses are inside the $500 per-item limit on First and the deductible is $0.
- Tests: A rule found only in the claims manual: losses to wild animals. Under the wording alone it reads as a theft with no police report


**CLM-1021, hard**: approve $1,053.00, where the handler said approve $825.00

- Model: Claim approved under Section 5 (Baggage and Personal Effects). The theft occurred on July 6, 2026, within the covered trip (July 2–12, 2026) in Buenos Aires. The police report was filed the same day (within the required 24 hours) and is attached, satisfying the theft documentation requirement. The bag was snatched from her shoulder, so the unattended-belongings exclusion does not apply, and no other exclusions are relevant.

Payout calculation per Section 10 (Premium plan; Electronics Plus add-on):
- Headphones $429.00: not within the Electronics Plus definition (phones, tablets, laptops, cameras/lenses, e-readers, smartwatches), so the standard $500 per-item limit applies — payable in full.
- E-reader $180.00: covered electronics under the add-on ($1,200 per-item limit) — payable in full.
- Handheld game console $399.00: not within the Electronics Plus definition, standard $500 per-item limit applies — payable in full.
- Canvas/leather tote $95.00: no receipt is required by the wording for baggage claims; the police report corroborates the loss — payable in full.
Total eligible: $1,103.00. No carrier or third-party payments to deduct. Less the $50 Premium deductible (applies once per baggage claim): $1,053.00. This is within the $1,500 plan maximum.
- Handler: Theft from the person, reported to the police the same afternoon. Electronics Plus raises the per-item limit only for the devices the wording lists: phones, tablets, laptops, cameras and their lenses, e-readers and smartwatches. The e-reader is on the list and is under the limit anyway. Headphones and a game console are not on the list, so each counts at the $300 per-item limit on Premium. The tote is under $100, so the customer's own figure is accepted. The $50 deductible comes off.
- Tests: An add-on with a defined scope: Electronics Plus covers a closed list of devices, and the claim form's category says electronics for items that are not on it


In [71]:

rows = []
for results in [free_results, agent_results]:
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)

,LLM,technique,accuracy,speed
0,minimaxai/minimax-m3.1-flash,LLM,3 / 5,13.6 seconds a claim
1,openrouter/free,Agent,3 / 5,86.0 seconds a claim
